# 03 — Non-convex regime: small CNN on CIFAR-10

Same optimizer set as in notebook 02, but the model is now a small CNN. The loss is no longer convex, so the theoretical guarantees of the course do **not** apply directly. The point of this notebook is to:

1. Observe empirically how each optimizer behaves on a non-convex loss.
2. Contrast with what we saw in the convex regime (notebook 02).
3. Provide material for the report's discussion: why first-order methods still work, why 2nd-order ones are out of reach, what "adaptive" methods like Adam buy us in practice.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import torch
import matplotlib.pyplot as plt
from src.train import train_cnn
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
# Keep epochs small for the initial comparison — bump to 30+ for the final run.
EPOCHS = 10
configs = [
    ('sgd',      dict(lr=1e-2, momentum=0.0)),
    ('momentum', dict(lr=1e-2, momentum=0.9)),
    ('nesterov', dict(lr=1e-2, momentum=0.9)),
    ('adagrad',  dict(lr=1e-2, momentum=0.0)),
    ('rmsprop',  dict(lr=1e-3, momentum=0.0)),
    ('adam',     dict(lr=1e-3, momentum=0.0)),
]
histories = {}
for name, kw in configs:
    print(f'==== {name} ====')
    h = train_cnn(
        optimizer=name, epochs=EPOCHS, batch_size=128,
        lr=kw['lr'], momentum=kw['momentum'],
        weight_decay=5e-4, augment=True, seed=0, device=device,
    )
    histories[name] = h

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name, h in histories.items():
    axes[0].plot(range(1, len(h.train_loss) + 1), h.train_loss, label=name)
    axes[1].plot(range(1, len(h.test_acc) + 1),   h.test_acc,   label=name)
    axes[2].plot(range(1, len(h.grad_norm) + 1),  h.grad_norm,  label=name)
axes[0].set(xlabel='epoch', ylabel='train loss', yscale='log', title='CNN — training loss')
axes[1].set(xlabel='epoch', ylabel='test accuracy',           title='CNN — test accuracy')
axes[2].set(xlabel='epoch', ylabel='|grad|', yscale='log',    title='CNN — gradient norm')
for ax in axes:
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()

## Notes for the report

- The gradient norm typically **does not converge to zero** on the CNN — this is one of the most striking visual differences with the convex regime, and a great hook to discuss convergence guarantees vs. their absence.
- Compare the *ranking* of optimizers in the convex regime vs here. The best convex optimizer (often Nesterov / L-BFGS) is rarely the best on the CNN, where adaptive methods (Adam) usually win early-epoch.
- Discuss generalization: training loss is not the only thing that matters. SGD+momentum often generalizes better than Adam on image tasks — a known empirical fact worth mentioning, with references.